[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brilliantbeaver/alexpose/blob/main/penny/gavd3/07_source_video_identity_audit.ipynb)

# 07. Audit what the frozen encoder remembers

Measure how much of the S-JEPA representation is gait and how much is source-video identity, then put the confounded sequence splits next to video-grouped evaluation.

**Research use only.** This tutorial does not diagnose a person or validate a clinical device.

**Run it:** locally, use `uv sync` then `uv run jupyter lab` from this folder. In Colab, use the badge and run the setup cell. Restart the kernel after changing `penny/gavd3/.env`.

**Keep the walk visible:** notebook 01 opens the source video and notebook 02 shows frame, bbox, and skeleton alignment. Revisit those views whenever an audit number looks surprising.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/brilliantbeaver/alexpose.git"

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "numpy", "pandas", "scipy", "scikit-learn", "matplotlib",
        "seaborn", "torch", "tqdm", "python-dotenv", "yt-dlp[default]",
        "opencv-python-headless", "mediapipe<1", "joblib", "pyarrow",
    ])
    clone_dir = Path("/content/alexpose")
    if not (clone_dir / ".git").exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)])
    os.chdir(clone_dir)


def find_project_root(start=None):
    env_root = os.getenv("ALEXPOSE_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if (candidate / ".git").exists() and (candidate / "data" / "gavd").exists():
            return candidate
        print(f"Ignoring invalid ALEXPOSE_ROOT: {candidate}")
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() and (candidate / "data" / "gavd").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
TUTORIAL_DIR = PROJECT_ROOT / "penny" / "gavd3"

try:
    from dotenv import load_dotenv
    load_dotenv(TUTORIAL_DIR / ".env", override=False)
    load_dotenv(PROJECT_ROOT / ".env", override=False)
except Exception:
    pass

MODE = os.getenv("GAVD3_MODE", "smoke").strip().lower()
if MODE not in {"smoke", "real"}:
    raise ValueError("GAVD3_MODE must be smoke or real")
if MODE == "smoke":
    print(
        "SMOKE MODE: hand-authored motions test code paths only. "
        "They have no pathophysiological or clinical validity."
    )

PREFERRED_ROOT = Path(
    os.getenv(
        "GAVD4_ROOT",
        "/Users/pmui/vaults/worldmodels/gait/skeleton-jepa/gavd4",
    )
).expanduser()

requested_data = os.getenv("GAVD4_DATA_DIR") or os.getenv("GAVD_DATA_GAVD_DIR")
if requested_data and Path(requested_data).expanduser().exists():
    DATA_GAVD_DIR = Path(requested_data).expanduser()
elif requested_data:
    print(f"Ignoring missing GAVD CSV path: {Path(requested_data).expanduser()}")
    if (PREFERRED_ROOT / "data-gavd").exists():
        DATA_GAVD_DIR = PREFERRED_ROOT / "data-gavd"
    else:
        DATA_GAVD_DIR = PROJECT_ROOT / "data" / "gavd"
elif (PREFERRED_ROOT / "data-gavd").exists():
    DATA_GAVD_DIR = PREFERRED_ROOT / "data-gavd"
else:
    DATA_GAVD_DIR = PROJECT_ROOT / "data" / "gavd"

requested_youtube = os.getenv("GAVD4_YOUTUBE_DIR") or os.getenv("GAVD_YOUTUBE_DIR")
if requested_youtube:
    YOUTUBE_DIR = Path(requested_youtube).expanduser()
elif PREFERRED_ROOT.exists():
    YOUTUBE_DIR = PREFERRED_ROOT / "youtube"
else:
    YOUTUBE_DIR = PROJECT_ROOT / "penny" / "gavd3" / "work" / "youtube"

CACHE_DIR = Path(
    os.getenv("GAVD3_CACHE_DIR", TUTORIAL_DIR / "work" / "cache")
).expanduser()
ARTIFACT_ROOT = Path(
    os.getenv("GAVD3_ARTIFACT_DIR", TUTORIAL_DIR / "work" / "artifacts")
).expanduser()
ARTIFACT_DIR = ARTIFACT_ROOT / MODE
POSE_DIR = ARTIFACT_DIR / "poses"

for folder in [CACHE_DIR, ARTIFACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR / "xdg-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
from IPython.display import SVG, display


def show_tutorial_svg(filename):
    '''Render a repository SVG reliably in local Jupyter and Colab.'''
    path = TUTORIAL_DIR / "images" / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing tutorial figure {path}. Clone the full alexpose repository."
        )
    display(SVG(filename=str(path)))

print(f"mode: {MODE}")
print(f"project: {PROJECT_ROOT}")
print(f"GAVD CSVs: {DATA_GAVD_DIR}")
print(f"YouTube cache: {YOUTUBE_DIR}")
print(f"artifacts: {ARTIFACT_DIR}")

## Why source identity matters

Notebook 06 showed that every test sequence shares its source video with training data. All 9 exp5 test videos also appear in training, and 3 test sequences come from the exact video used for normal-only pretraining. That makes the 61.9 percent five-class score a transductive number: it describes how well the model recognizes clips it has effectively seen before, not how well it would recognize a new person, camera, or clinic.

The same failure mode has a name in the biosignal foundation-model literature: subject-identity leakage. EEG models that look excellent on random splits can partly work by decoding who the recording came from rather than what the brain is doing. For pose extracted from video, the analogous confound is source-video identity. One YouTube upload shares the same person, clothing, background, camera angle, compression, and pose-detector noise pattern. If the frozen latents can predict which upload a clip came from, then a disease classifier built on those latents is partly an upload matcher.

This notebook turns that worry into four measurements:

1. a video-ID probe: how well the pooled latents identify the source videos;
2. a video-grouped five-class evaluation with correct majority-class controls;
3. the generalization gap: sequence-split accuracy minus source-grouped accuracy;
4. a source-grouped one-versus-normal check where the data allows it.

One nuance matters. Gait is personally identifying, and gait recognition is a legitimate research field. Some video-identity content in a gait representation is expected, because different people really do walk differently. The audit therefore compares the pretrained encoder against two baselines built from the same clips: pooled raw coordinates and pose-missingness statistics. If the pretrained encoder carries no more video identity than the raw pose does, the identity content is mostly the walking pattern itself. If it carries more, some of it may be recording artifacts.

In [ ]:
show_tutorial_svg("08_evaluation_protocol.svg")

In [ ]:
BLAZEPOSE_33 = [
    "NOSE", "LEFT_EYE_INNER", "LEFT_EYE", "LEFT_EYE_OUTER",
    "RIGHT_EYE_INNER", "RIGHT_EYE", "RIGHT_EYE_OUTER", "LEFT_EAR",
    "RIGHT_EAR", "MOUTH_LEFT", "MOUTH_RIGHT", "LEFT_SHOULDER",
    "RIGHT_SHOULDER", "LEFT_ELBOW", "RIGHT_ELBOW", "LEFT_WRIST",
    "RIGHT_WRIST", "LEFT_PINKY", "RIGHT_PINKY", "LEFT_INDEX",
    "RIGHT_INDEX", "LEFT_THUMB", "RIGHT_THUMB", "LEFT_HIP",
    "RIGHT_HIP", "LEFT_KNEE", "RIGHT_KNEE", "LEFT_ANKLE",
    "RIGHT_ANKLE", "LEFT_HEEL", "RIGHT_HEEL", "LEFT_FOOT_INDEX",
    "RIGHT_FOOT_INDEX",
]
MASK_KEYPOINTS = [11, 12, 23, 24, 25, 26, 27, 28, 31, 32]
assert [BLAZEPOSE_33[i] for i in MASK_KEYPOINTS] == [
    "LEFT_SHOULDER", "RIGHT_SHOULDER", "LEFT_HIP", "RIGHT_HIP",
    "LEFT_KNEE", "RIGHT_KNEE", "LEFT_ANKLE", "RIGHT_ANKLE",
    "LEFT_FOOT_INDEX", "RIGHT_FOOT_INDEX",
]



CONDITIONS = ["normal", "parkinsons", "stroke", "cerebralpalsy", "myopathic"]

In [ ]:
def synthetic_gait_sequence(condition="normal", frames=64, seed=0):
    '''Create a code-path fixture, not a physiological disease simulation.'''
    rng = np.random.default_rng(seed)
    phase = np.linspace(0.0, 4.0 * np.pi, frames, endpoint=False)
    seq = np.zeros((frames, 33, 4), dtype=np.float32)
    seq[..., 3] = 1.0
    base = {
        11: (0.42, 0.28), 12: (0.58, 0.28),
        23: (0.45, 0.52), 24: (0.55, 0.52),
        25: (0.44, 0.70), 26: (0.56, 0.70),
        27: (0.43, 0.89), 28: (0.57, 0.89),
        29: (0.42, 0.92), 30: (0.58, 0.92),
        31: (0.39, 0.94), 32: (0.61, 0.94),
    }
    for joint, (x, y) in base.items():
        seq[:, joint, 0] = x
        seq[:, joint, 1] = y
    amplitude = 0.045
    lift = 0.025
    if condition == "parkinsons":
        amplitude *= 0.45
        lift *= 0.45
    if condition == "myopathic":
        seq[:, [11, 12], 0] += 0.03 * np.sin(phase)[:, None]
        seq[:, [23, 24], 0] += 0.018 * np.sin(phase)[:, None]
    for joint, knee, foot, offset in [(27, 25, 31, 0.0), (28, 26, 32, np.pi)]:
        wave = np.sin(phase + offset)
        if condition == "stroke" and joint == 27:
            wave = 0.35 * wave
        if condition == "cerebralpalsy":
            seq[:, knee, 1] -= 0.045
            seq[:, joint, 1] -= 0.02
        seq[:, joint, 0] += amplitude * wave
        seq[:, knee, 0] += 0.4 * amplitude * wave
        seq[:, foot, 0] += amplitude * wave
        seq[:, joint, 1] -= lift * np.maximum(wave, 0.0)
        seq[:, foot, 1] -= 0.7 * lift * np.maximum(wave, 0.0)
    seq[..., :3] += rng.normal(0.0, 0.0025, seq[..., :3].shape)
    return seq


def synthetic_corpus(conditions=None, n_per_condition=10, frames=64, seed=42):
    if conditions is None:
        conditions = [
            "normal", "parkinsons", "stroke", "cerebralpalsy", "myopathic"
        ]
    records = []
    counter = 0
    for condition in conditions:
        for sample in range(n_per_condition):
            records.append({
                "condition": condition,
                "sequence_id": f"smoke_{condition}_{sample:03d}",
                "video_id": f"smoke_video_{condition}_{sample // 2:02d}",
                "sequence": synthetic_gait_sequence(
                    condition=condition,
                    frames=frames,
                    seed=seed + counter,
                ),
            })
            counter += 1
    return records

In [ ]:
def interpolate_low_visibility(sequence, threshold=0.45, max_gap=4):
    '''Fill only short internal gaps and preserve the original validity mask.

    Long gaps and sequence ends are never extrapolated. Their coordinates remain
    missing until center_and_scale converts them to an explicit zero sentinel.
    They can never become S-JEPA prediction targets.
    '''
    sequence = np.asarray(sequence, dtype=np.float32).copy()
    if sequence.ndim != 3 or sequence.shape[1:] != (33, 4):
        raise ValueError(f"Expected [T, 33, 4], received {sequence.shape}")
    visibility = np.nan_to_num(sequence[..., 3], nan=0.0)
    finite = np.isfinite(sequence[..., :3]).all(axis=-1)
    valid = (visibility >= threshold) & finite
    filled = valid.copy()
    for joint in range(33):
        observed = np.flatnonzero(valid[:, joint])
        for left, right in zip(observed[:-1], observed[1:]):
            gap = int(right - left - 1)
            if not 0 < gap <= max_gap:
                continue
            fraction = (
                np.arange(1, gap + 1, dtype=np.float32) / (gap + 1)
            )[:, None]
            sequence[left + 1:right, joint, :3] = (
                sequence[left, joint, :3][None, :] * (1.0 - fraction)
                + sequence[right, joint, :3][None, :] * fraction
            )
            filled[left + 1:right, joint] = True
        sequence[~filled[:, joint], joint, :3] = np.nan
    sequence[..., 3] = visibility
    return sequence, valid


def center_and_scale(sequence, eps=1e-6):
    sequence = np.asarray(sequence, dtype=np.float32).copy()
    xyz = sequence[..., :3]
    left_hip, right_hip = xyz[:, 23], xyz[:, 24]
    left_ok = np.isfinite(left_hip).all(axis=1)
    right_ok = np.isfinite(right_hip).all(axis=1)
    pelvis = np.full((len(xyz), 3), np.nan, dtype=np.float32)
    pelvis[left_ok & right_ok] = 0.5 * (
        left_hip[left_ok & right_ok] + right_hip[left_ok & right_ok]
    )
    pelvis[left_ok & ~right_ok] = left_hip[left_ok & ~right_ok]
    pelvis[right_ok & ~left_ok] = right_hip[right_ok & ~left_ok]
    pelvis_ok = np.isfinite(pelvis).all(axis=1)
    fallback = np.median(pelvis[pelvis_ok], axis=0) if pelvis_ok.any() else np.zeros(3)
    pelvis[~np.isfinite(pelvis).all(axis=1)] = fallback
    xyz = xyz - pelvis[:, None, :]
    shoulder_width = np.linalg.norm(xyz[:, 11, :2] - xyz[:, 12, :2], axis=-1)
    hip_width = np.linalg.norm(xyz[:, 23, :2] - xyz[:, 24, :2], axis=-1)
    body_scale = np.nanmedian(np.maximum(shoulder_width, hip_width))
    if not np.isfinite(body_scale) or body_scale < eps:
        body_scale = 1.0
    sequence[..., :3] = np.nan_to_num(
        xyz / body_scale, nan=0.0, posinf=0.0, neginf=0.0
    )
    return np.nan_to_num(sequence, nan=0.0, posinf=0.0, neginf=0.0)


def temporal_resize(array, frames):
    array = np.asarray(array)
    if len(array) == frames:
        return array.copy()
    if len(array) < 2:
        return np.repeat(array, frames, axis=0)
    old_t = np.linspace(0.0, 1.0, len(array))
    new_t = np.linspace(0.0, 1.0, frames)
    flat = array.reshape(len(array), -1)
    resized = np.stack(
        [np.interp(new_t, old_t, flat[:, i]) for i in range(flat.shape[1])],
        axis=1,
    )
    return resized.reshape(frames, *array.shape[1:]).astype(array.dtype)


def prepare_sequence(
    sequence,
    frames=64,
    visibility_threshold=0.45,
    max_gap=4,
):
    cleaned, valid = interpolate_low_visibility(
        sequence, visibility_threshold, max_gap=max_gap
    )
    cleaned = center_and_scale(cleaned)
    cleaned = temporal_resize(cleaned, frames)
    valid = temporal_resize(valid.astype(np.float32), frames) >= 0.5
    return cleaned[..., :3].astype(np.float32), valid.astype(bool)

In [ ]:
import copy
import math
import torch
from torch import nn


class SkeletonPatchEncoder(nn.Module):
    def __init__(
        self,
        frames=64,
        joints=33,
        coordinate_dim=3,
        segment_length=4,
        embed_dim=64,
        depth=2,
        heads=4,
        dropout=0.0,
    ):
        super().__init__()
        if frames % segment_length:
            raise ValueError("frames must be divisible by segment_length")
        self.frames = frames
        self.joints = joints
        self.coordinate_dim = coordinate_dim
        self.segment_length = segment_length
        self.segments = frames // segment_length
        self.embed_dim = embed_dim
        self.patch_embed = nn.Linear(segment_length * coordinate_dim, embed_dim)
        self.time_pos = nn.Parameter(torch.randn(self.segments, embed_dim) * 0.02)
        self.joint_pos = nn.Parameter(torch.randn(joints, embed_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)

    def patchify(self, x):
        batch, frames, joints, channels = x.shape
        expected = (self.frames, self.joints, self.coordinate_dim)
        if (frames, joints, channels) != expected:
            raise ValueError(f"Expected [B, {expected}], received {x.shape}")
        patches = x.reshape(
            batch, self.segments, self.segment_length, joints, channels
        )
        patches = patches.permute(0, 1, 3, 2, 4).contiguous()
        return patches.flatten(3)

    def positioned_tokens(self, x):
        tokens = self.patch_embed(self.patchify(x))
        return (
            tokens
            + self.time_pos[None, :, None, :]
            + self.joint_pos[None, None, :, :]
        )

    def forward(self, x, keep_mask=None):
        tokens = self.positioned_tokens(x)
        batch = len(tokens)
        flat = tokens.reshape(batch, self.segments * self.joints, self.embed_dim)
        if keep_mask is not None:
            keep_mask = keep_mask.reshape(batch, -1)
            kept_per_sample = keep_mask.sum(dim=1)
            if not torch.equal(kept_per_sample, kept_per_sample[:1].expand_as(kept_per_sample)):
                raise ValueError("Each sample must keep the same number of tokens")
            flat = flat[keep_mask].reshape(batch, int(kept_per_sample[0]), self.embed_dim)
        return self.norm(self.blocks(flat))


class SkeletonPredictor(nn.Module):
    def __init__(
        self,
        segments,
        joints,
        encoder_dim=64,
        predictor_dim=64,
        depth=2,
        heads=4,
        dropout=0.0,
    ):
        super().__init__()
        self.segments = segments
        self.joints = joints
        self.encoder_to_predictor = nn.Linear(encoder_dim, predictor_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, predictor_dim))
        nn.init.normal_(self.mask_token, std=0.02)
        self.time_pos = nn.Parameter(torch.randn(segments, predictor_dim) * 0.02)
        self.joint_pos = nn.Parameter(torch.randn(joints, predictor_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=predictor_dim,
            nhead=heads,
            dim_feedforward=predictor_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.blocks = nn.TransformerEncoder(layer, num_layers=depth)
        self.norm = nn.LayerNorm(predictor_dim)
        self.output = nn.Linear(predictor_dim, encoder_dim)

    def forward(self, visible_features, target_mask):
        batch = len(visible_features)
        target_mask = target_mask.reshape(batch, self.segments * self.joints)
        visible_mask = ~target_mask
        visible = self.encoder_to_predictor(visible_features)
        full = self.mask_token.expand(
            batch, self.segments * self.joints, -1
        ).clone()
        full[visible_mask] = visible.reshape(-1, visible.shape[-1])
        positions = (
            self.time_pos[:, None, :] + self.joint_pos[None, :, :]
        ).reshape(1, self.segments * self.joints, -1)
        full = full + positions
        predicted = self.output(self.norm(self.blocks(full)))
        return predicted[target_mask].reshape(batch, -1, predicted.shape[-1])


class SJEPAGait(nn.Module):
    def __init__(
        self,
        frames=64,
        joints=33,
        coordinate_dim=3,
        segment_length=4,
        embed_dim=64,
        encoder_depth=2,
        predictor_depth=2,
        heads=4,
    ):
        super().__init__()
        self.view_encoder = SkeletonPatchEncoder(
            frames, joints, coordinate_dim, segment_length,
            embed_dim, encoder_depth, heads,
        )
        self.target_encoder = copy.deepcopy(self.view_encoder)
        for parameter in self.target_encoder.parameters():
            parameter.requires_grad_(False)
        self.predictor = SkeletonPredictor(
            self.view_encoder.segments,
            joints,
            embed_dim,
            embed_dim,
            predictor_depth,
            heads,
        )
        self.register_buffer("target_center", torch.zeros(embed_dim))

    def forward(self, view, target, target_mask):
        visible_features = self.view_encoder(view, keep_mask=~target_mask)
        predicted = self.predictor(visible_features, target_mask)
        with torch.no_grad():
            all_targets = self.target_encoder(target)
            flat_mask = target_mask.reshape(len(target), -1)
            selected = all_targets[flat_mask].reshape(
                len(target), -1, all_targets.shape[-1]
            )
        return predicted, selected

    @torch.no_grad()
    def update_target(self, momentum):
        for target_parameter, view_parameter in zip(
            self.target_encoder.parameters(), self.view_encoder.parameters()
        ):
            target_parameter.mul_(momentum).add_(
                view_parameter, alpha=1.0 - momentum
            )

    @torch.no_grad()
    def update_center(self, targets, beta=0.9):
        batch_center = targets.mean(dim=(0, 1))
        self.target_center.mul_(beta).add_(batch_center, alpha=1.0 - beta)


def sjepa_cross_entropy(
    predicted,
    targets,
    center,
    predictor_temperature=0.10,
    target_temperature=0.06,
):
    target_prob = torch.softmax(
        (targets - center[None, None, :]) / target_temperature,
        dim=-1,
    ).detach()
    prediction_log_prob = torch.log_softmax(
        predicted / predictor_temperature,
        dim=-1,
    )
    return -(target_prob * prediction_log_prob).sum(dim=-1).mean()


def cosine_ema(step, total_steps, start=0.996, end=1.0):
    progress = min(max(step / max(total_steps - 1, 1), 0.0), 1.0)
    return end - (end - start) * (math.cos(math.pi * progress) + 1.0) / 2.0


LEFT_RIGHT_PAIRS = [
    (1, 4), (2, 5), (3, 6), (7, 8), (9, 10), (11, 12),
    (13, 14), (15, 16), (17, 18), (19, 20), (21, 22),
    (23, 24), (25, 26), (27, 28), (29, 30), (31, 32),
]


def geometric_view(
    x,
    max_degrees=8.0,
    translate=0.03,
    flip_probability=0.0,
):
    """Apply one sequence-wide transform per sample.

    Rotation is around the relative vertical y axis, so x and z are mixed.
    Flip defaults to off because laterality can matter for stroke. If enabled,
    coordinates are reflected and every left-right landmark pair is swapped.
    """
    view = x.clone()
    present = view.abs().sum(dim=-1) > 1e-8
    batch = len(view)
    angles = (
        torch.rand(batch, device=x.device) * 2.0 - 1.0
    ) * math.radians(max_degrees)
    cosine, sine = torch.cos(angles), torch.sin(angles)
    original_x = view[..., 0].clone()
    original_z = view[..., 2].clone()
    rotated_x = cosine[:, None, None] * original_x + sine[:, None, None] * original_z
    rotated_z = -sine[:, None, None] * original_x + cosine[:, None, None] * original_z
    view[..., 0] = rotated_x
    view[..., 2] = rotated_z
    offsets = (torch.rand(batch, 1, 1, 2, device=x.device) * 2.0 - 1.0) * translate
    view[..., :2] += offsets
    if flip_probability > 0:
        flip = torch.rand(batch, device=x.device) < flip_probability
        for batch_index in torch.where(flip)[0].tolist():
            view[batch_index, ..., 0] *= -1.0
            original = view[batch_index].clone()
            original_present = present[batch_index].clone()
            for left, right in LEFT_RIGHT_PAIRS:
                view[batch_index, :, left] = original[:, right]
                view[batch_index, :, right] = original[:, left]
                present[batch_index, :, left] = original_present[:, right]
                present[batch_index, :, right] = original_present[:, left]
    view = view.masked_fill(~present[..., None], 0.0)
    return view

In [ ]:
def pose_records_from_cache(pose_dir=POSE_DIR, conditions=CONDITIONS):
    records = []
    for condition in conditions:
        folder = Path(pose_dir) / condition
        for path in sorted(folder.glob("*.npz")):
            data = np.load(path, allow_pickle=False)
            required = {
                "sequence", "sequence_id", "video_id", "condition",
                "frame_numbers", "crop_bounds", "fps", "source_csv",
                "source_video", "pose_model", "pose_model_sha256",
                "extraction_version",
            }
            missing = required.difference(data.files)
            if missing:
                raise ValueError(
                    f"Stale pose cache {path} is missing {sorted(missing)}. "
                    "Re-extract it with notebook 02."
                )
            sequence = data["sequence"].astype(np.float32)
            if sequence.ndim != 3 or sequence.shape[1:] != (33, 4):
                raise ValueError(f"Bad pose shape in {path}: {sequence.shape}")
            stored_condition = str(data["condition"].item())
            if stored_condition != condition:
                raise ValueError(
                    f"Pose condition {stored_condition} does not match folder {condition}"
                )
            if len(data["frame_numbers"]) != len(sequence):
                raise ValueError(f"Frame and pose lengths differ in {path}")
            records.append({
                "condition": condition,
                "sequence_id": str(data["sequence_id"].item()),
                "video_id": str(data["video_id"].item()),
                "source_video": str(data["source_video"].item()),
                "fps": float(data["fps"].item()),
                "extraction_version": str(data["extraction_version"].item()),
                "pose_model_sha256": str(data["pose_model_sha256"].item()),
                "sequence": sequence,
                "path": str(path),
            })
    return records


def load_records_for_mode(conditions=CONDITIONS, smoke_per_condition=10, frames=64):
    if MODE == "smoke":
        records = synthetic_corpus(
            conditions=conditions,
            n_per_condition=smoke_per_condition,
            frames=frames,
        )
        print(f"Explicit smoke corpus: {len(records)} synthetic sequences")
        return records
    records = pose_records_from_cache(conditions=conditions)
    counts = pd.Series([r["condition"] for r in records]).value_counts()
    missing = [condition for condition in conditions if counts.get(condition, 0) == 0]
    if missing:
        raise FileNotFoundError(
            f"Real mode requires cached pose sequences for {missing}. "
            "Run notebook 02 first."
        )
    print(f"Real pose corpus: {len(records)} sequences")
    return records

In [ ]:
def masked_mean_std(tokens, mask):
    weights = torch.as_tensor(
        mask, dtype=tokens.dtype, device=tokens.device
    ).unsqueeze(-1)
    denominator = weights.sum(dim=1).clamp_min(1.0)
    mean = (tokens * weights).sum(dim=1) / denominator
    variance = (
        (tokens - mean[:, None, :]).square() * weights
    ).sum(dim=1) / denominator
    return mean, variance.clamp_min(0.0).sqrt()


@torch.no_grad()
def pooled_embeddings(model, arrays, validity, batch_size=8):
    vectors = []
    segments = model.target_encoder.segments
    segment_length = model.target_encoder.segment_length
    dimension = model.target_encoder.embed_dim
    for start in range(0, len(arrays), batch_size):
        batch = torch.tensor(
            arrays[start:start + batch_size],
            dtype=torch.float32,
        )
        tokens = model.target_encoder(batch).reshape(
            len(batch), segments, 33, dimension
        )
        valid_patch = np.asarray(
            validity[start:start + batch_size], dtype=bool
        ).reshape(
            len(batch), segments, segment_length, 33
        ).all(axis=2)
        global_tokens = tokens.reshape(len(batch), -1, dimension)
        neuro_tokens = tokens[:, :, MASK_KEYPOINTS].reshape(
            len(batch), -1, dimension
        )
        global_mean, global_std = masked_mean_std(
            global_tokens, valid_patch.reshape(len(batch), -1)
        )
        neuro_mean, neuro_std = masked_mean_std(
            neuro_tokens,
            valid_patch[:, :, MASK_KEYPOINTS].reshape(len(batch), -1),
        )
        vector = torch.cat(
            [
                global_mean,
                global_std,
                neuro_mean,
                neuro_std,
            ],
            dim=1,
        )
        vectors.append(vector.cpu())
    return torch.cat(vectors).numpy()


## Load the frozen encoder and pool every sequence

The audit reuses the exact frozen target encoder from notebook 04 and the exact 384-value pooling from notebooks 05 and 06: mean and standard deviation over all valid tokens, plus mean and standard deviation over the ten neurologic landmarks.

Real mode refuses to continue without the checkpoint. Smoke mode builds a small, randomly initialized encoder so every audit step still runs end to end.

In [ ]:
import torch

checkpoint_path = ARTIFACT_DIR / "sjepa_normal.pt"
if MODE == "real":
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Missing {checkpoint_path}. Run notebook 04 in {MODE} mode first."
        )
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if checkpoint["mode"] != MODE:
        raise ValueError(f"Checkpoint mode {checkpoint['mode']} does not match {MODE}")
    if checkpoint["mask_keypoints"] != MASK_KEYPOINTS:
        raise ValueError("Checkpoint mask set does not match this tutorial")
    model = SJEPAGait(**checkpoint["config"])
    model.load_state_dict(checkpoint["model_state"])
    model.eval()
    FRAMES = checkpoint["config"]["frames"]
    SEGMENT_LENGTH = checkpoint["config"]["segment_length"]
    print("loaded checkpoint:", checkpoint_path)
    print("dataset fingerprint:", checkpoint["dataset_fingerprint"])
else:
    FRAMES, SEGMENT_LENGTH = 32, 4
    model = SJEPAGait(
        frames=FRAMES, segment_length=SEGMENT_LENGTH, embed_dim=32,
        encoder_depth=1, predictor_depth=1, heads=4,
    )
    model.eval()
    print(
        "SMOKE MODE: using a randomly initialized compact encoder. "
        "Smoke numbers exercise the audit code path only."
    )

records = load_records_for_mode(
    conditions=CONDITIONS, smoke_per_condition=10, frames=FRAMES
)
prepared = [prepare_sequence(record["sequence"], frames=FRAMES) for record in records]
all_xyz = np.stack([item[0] for item in prepared])
all_valid = np.stack([item[1] for item in prepared])
labels = np.asarray([record["condition"] for record in records])
sequence_ids = np.asarray([record["sequence_id"] for record in records])
video_ids = np.asarray([record["video_id"] for record in records])
print("sequences:", len(records), "| source videos:", len(set(video_ids)))
print(pd.Series(labels).value_counts().reindex(CONDITIONS))

embeddings = pooled_embeddings(model, all_xyz, all_valid)
print("pooled embeddings:", embeddings.shape)
assert np.isfinite(embeddings).all()

# Control arm: an untrained encoder with the identical architecture and
# pooling. Whatever identity content pretraining added beyond the raw pose
# shows up as the gap between this arm and the frozen encoder.
encoder_config = checkpoint["config"] if MODE == "real" else {
    "frames": FRAMES, "joints": 33, "coordinate_dim": 3,
    "segment_length": SEGMENT_LENGTH, "embed_dim": 32,
    "encoder_depth": 1, "predictor_depth": 1, "heads": 4,
}
torch.manual_seed(42)
random_model = SJEPAGait(**encoder_config)
random_model.eval()
random_embeddings = pooled_embeddings(random_model, all_xyz, all_valid)
print("untrained encoder embeddings:", random_embeddings.shape)
assert np.isfinite(random_embeddings).all()


def raw_pooled_embeddings(arrays, validity, frames, segment_length, keypoints):
    '''Baseline 1: the same pooling arithmetic on raw patch coordinates.'''
    vectors = []
    segments = frames // segment_length
    for start in range(0, len(arrays), 8):
        batch = torch.tensor(arrays[start:start + 8], dtype=torch.float32)
        batch_valid = np.asarray(validity[start:start + 8], dtype=bool)
        patches = (
            batch.reshape(len(batch), segments, segment_length, 33, 3)
            .permute(0, 1, 3, 2, 4).contiguous().flatten(3)
        )
        valid_patch = batch_valid.reshape(
            len(batch), segments, segment_length, 33
        ).all(axis=2)
        global_tokens = patches.reshape(len(batch), -1, patches.shape[-1])
        neuro_tokens = patches[:, :, keypoints].reshape(
            len(batch), -1, patches.shape[-1]
        )
        global_mean, global_std = masked_mean_std(
            global_tokens, valid_patch.reshape(len(batch), -1)
        )
        neuro_mean, neuro_std = masked_mean_std(
            neuro_tokens,
            valid_patch[:, :, keypoints].reshape(len(batch), -1),
        )
        vectors.append(
            torch.cat([global_mean, global_std, neuro_mean, neuro_std], dim=1).cpu()
        )
    return torch.cat(vectors).numpy()


raw_pooled = raw_pooled_embeddings(
    all_xyz, all_valid, FRAMES, SEGMENT_LENGTH, MASK_KEYPOINTS
)
print("raw pose pooling baseline:", raw_pooled.shape)


def missingness_signature(sequence, threshold=0.45):
    '''Baseline 2: pose-detector failure statistics per sequence.'''
    visibility = np.nan_to_num(sequence[..., 3], nan=0.0)
    finite = np.isfinite(sequence[..., :3]).all(axis=-1)
    valid = (visibility >= threshold) & finite
    per_joint_visible = valid.mean(axis=0)
    confidence_sum = np.where(valid, visibility, 0.0).sum(axis=0)
    per_joint_mean_confidence = confidence_sum / valid.sum(axis=0).clip(min=1)
    per_joint_mean_gap = []
    for joint in range(33):
        observed = np.flatnonzero(valid[:, joint])
        if len(observed) < 2:
            per_joint_mean_gap.append(float(len(valid)))
            continue
        per_joint_mean_gap.append(float(np.diff(observed).mean() - 1.0))
    return np.concatenate([
        per_joint_visible,
        per_joint_mean_confidence,
        np.asarray(per_joint_mean_gap),
        [float(valid.mean())],
    ])


missingness_features = np.stack([
    missingness_signature(record["sequence"]) for record in records
])
print("missingness baseline:", missingness_features.shape)

## Audit 1: the video-ID probe

Fit a Random Forest that predicts which source video a clip came from, using only the pooled representation. Repeated 70/30 sequence splits (stratification unavailable with singleton videos) keep most videos on both sides, exactly the condition a confounded classifier enjoys, and the mean and standard deviation across three split seeds are reported. A fully video-disjoint probe would be trivially chance, because an unseen video is an unseen class, so it is not the right tool here.

Compare three representations built from the same clips:

| Representation | What it measures |
|---|---|
| Frozen S-JEPA 384-d | The pretrained latent summary |
| Untrained encoder 384-d | The identical architecture and pooling with random weights, seeded for reproducibility |
| Raw pose pooling 48-d | The same pooling math applied to unprojected coordinates |
| Missingness 100-d | How often the pose detector failed, per joint |

The chance line is one over the number of source videos. Three comparisons matter: pretrained versus raw pose (did pretraining add identity content beyond the pose?), pretrained versus untrained encoder (did pretraining add identity content beyond the architecture?), and all arms versus missingness (how much identity is detector failure?).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# The probe asks: given training clips from the videos, can the latents recognize
# which video a held-out clip belongs to? A stratified sequence split keeps every
# video on both sides, exactly the condition a confounded classifier enjoys. A
# fully video-disjoint probe would be trivially chance, because an unseen video
# is an unseen class. Because several videos contribute only one or two clips, neither
# k-fold nor stratified splitting is available, so the probe uses repeated
# plain 70/30 splits and reports the mean and standard deviation across seeds.
video_codes, unique_videos = pd.factorize(video_ids)
PROBE_SEEDS = [42, 43, 44]


def probe_video_identity(features):
    scores = []
    for seed in PROBE_SEEDS:
        # No stratification: two videos contribute a single clip, and
        # sklearn stratification requires at least two members per class.
        fit_index, held_index = train_test_split(
            np.arange(len(video_codes)),
            train_size=0.70,
            random_state=seed,
        )
        model = make_pipeline(
            StandardScaler(),
            RandomForestClassifier(
                n_estimators=100, random_state=42, class_weight="balanced"
            ),
        )
        model.fit(features[fit_index], video_codes[fit_index])
        prediction = model.predict(features[held_index])
        scores.append(float((prediction == video_codes[held_index]).mean()))
    return np.asarray(scores)


probe_results = {
    "frozen S-JEPA 384-d": probe_video_identity(embeddings),
    "untrained encoder 384-d": probe_video_identity(random_embeddings),
    "raw pose pooling 48-d": probe_video_identity(raw_pooled),
    "missingness 100-d": probe_video_identity(missingness_features),
}
probe_table = pd.DataFrame({
    "representation": list(probe_results.keys()),
    "mean_video_id_accuracy": [
        float(scores.mean()) for scores in probe_results.values()
    ],
    "accuracy_std": [
        float(scores.std(ddof=1)) for scores in probe_results.values()
    ],
})
probe_table["chance_1_of_n_videos"] = 1.0 / len(unique_videos)
display(probe_table)
print(
    "Probe caveats: (1) videos with one or two clips can land entirely in "
    "a test split, where their class is unseen and cannot be predicted, so "
    "the probe is a slight lower bound on separability; (2) the Random "
    "Forest uses class_weight balanced, which trades away majority-class "
    "accuracy and makes this probe conservative; (3) three split seeds are "
    "a spread, not a confidence interval. Run more seeds before quoting "
    "arm differences smaller than one standard deviation."
)

## Audit 2: sequence splits versus video-grouped splits

Notebook 06's fixed 70/30 sequence split reproduces here so the gap has a baseline. Then the same classifier runs twice under GroupKFold with identical fold machinery: once grouped by sequence and once grouped by source video. Only the grouping unit changes, so the difference between the two grouped lanes is a diagnostic that removes fold-size and retraining differences; fold class balance and clip composition can still differ between the lanes, so treat the matched gap as fold-controlled evidence, not a causal attribution.

The difference between the 70/30 lane and the video-grouped lane is the generalization gap; the difference between the sequence-grouped and video-grouped lanes is the matched grouping gap, which is the cleaner attribution. The correct majority-class control is computed in code for every lane. Note that the 0.294 majority figure in the earlier paper draft is not reproducible from the notebooks: on the 21-row exp5 test the best constant predictor scores 7 of 21, which is 0.333, and on this 29-row test it scores 14 of 29, which is 0.483.

In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import GroupKFold, train_test_split


def make_rf(seed=42):
    return make_pipeline(
        StandardScaler(),
        RandomForestClassifier(
            n_estimators=100, max_depth=5, max_features="sqrt",
            bootstrap=True, class_weight="balanced", random_state=seed,
        ),
    )


all_index = np.arange(len(labels))
train_index, test_index = train_test_split(
    all_index, train_size=0.70, random_state=42, stratify=labels
)
sequence_model = make_rf()
sequence_model.fit(embeddings[train_index], labels[train_index])
sequence_prediction = sequence_model.predict(embeddings[test_index])
sequence_metrics = {
    "accuracy": float(accuracy_score(labels[test_index], sequence_prediction)),
    "balanced_accuracy": float(
        balanced_accuracy_score(labels[test_index], sequence_prediction)
    ),
    "macro_f1": float(
        f1_score(labels[test_index], sequence_prediction,
                 average="macro", zero_division=0)
    ),
}
sequence_majority = pd.Series(labels[train_index]).mode().iloc[0]
sequence_metrics["majority_accuracy"] = float(
    (labels[test_index] == sequence_majority).mean()
)
print("sequence lane:", sequence_metrics)
print("sequence-lane test rows:", len(test_index))

group_splitter = GroupKFold(n_splits=6)
group_fold_rows = []
for fold, (fit_index, held_out_index) in enumerate(
    group_splitter.split(embeddings, groups=video_ids)
):
    grouped_model = make_rf(seed=42 + fold)
    grouped_model.fit(embeddings[fit_index], labels[fit_index])
    grouped_prediction = grouped_model.predict(embeddings[held_out_index])
    fold_majority = pd.Series(labels[fit_index]).mode().iloc[0]
    group_fold_rows.append({
        "fold": fold,
        "test_sequences": int(len(held_out_index)),
        "test_videos": int(len(set(video_ids[held_out_index]))),
        "accuracy": float(
            accuracy_score(labels[held_out_index], grouped_prediction)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(labels[held_out_index], grouped_prediction)
        ),
        "macro_f1": float(
            f1_score(labels[held_out_index], grouped_prediction,
                     average="macro", zero_division=0)
        ),
        "majority_accuracy": float(
            (labels[held_out_index] == fold_majority).mean()
        ),
    })
group_fold_table = pd.DataFrame(group_fold_rows)
display(group_fold_table)

generalization_gap = (
    sequence_metrics["accuracy"] - group_fold_table["accuracy"].mean()
)
print("generalization gap (sequence minus grouped):", round(generalization_gap, 4))

# Matched control lane: GroupKFold over SEQUENCES with the identical fold
# machinery. Only the grouping unit changes, so the difference between the
# two grouped lanes is attributable to source-video structure, not to fold
# size, class balance, or retraining.
sequence_splitter = GroupKFold(n_splits=6)
sequence_fold_rows = []
for fold, (fit_index, held_out_index) in enumerate(
    sequence_splitter.split(embeddings, groups=sequence_ids)
):
    sequence_grouped_model = make_rf(seed=42 + fold)
    sequence_grouped_model.fit(embeddings[fit_index], labels[fit_index])
    sequence_grouped_prediction = sequence_grouped_model.predict(
        embeddings[held_out_index]
    )
    fold_majority = pd.Series(labels[fit_index]).mode().iloc[0]
    sequence_fold_rows.append({
        "fold": fold,
        "test_sequences": int(len(held_out_index)),
        "accuracy": float(
            accuracy_score(labels[held_out_index], sequence_grouped_prediction)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                labels[held_out_index], sequence_grouped_prediction
            )
        ),
        "macro_f1": float(
            f1_score(
                labels[held_out_index], sequence_grouped_prediction,
                average="macro", zero_division=0,
            )
        ),
        "majority_accuracy": float(
            (labels[held_out_index] == fold_majority).mean()
        ),
    })
sequence_fold_table = pd.DataFrame(sequence_fold_rows)
display(sequence_fold_table)

matched_gap = (
    sequence_fold_table["accuracy"].mean()
    - group_fold_table["accuracy"].mean()
)
print(
    "matched grouped lanes: sequence-grouped",
    round(sequence_fold_table["accuracy"].mean(), 4),
    "vs video-grouped",
    round(group_fold_table["accuracy"].mean(), 4),
    "matched gap",
    round(matched_gap, 4),
)

## Audit 3: pretraining-video overlap

The normal-only pretraining used one video. Any test clip from that video is not a test of generalization to unseen walking, even under a grouped split. Count those clips in each lane.

In [ ]:
normal_videos = sorted(set(video_ids[labels == "normal"]))
test_video_set = set(video_ids[test_index])
shared_videos = sorted(test_video_set & set(video_ids[train_index]))
pretrain_overlap_count = int(np.isin(video_ids[test_index], normal_videos).sum())
print(
    "sequence lane: test videos shared with train:",
    len(shared_videos), "of", len(test_video_set),
)
print(
    "sequence lane: test sequences from the pretraining video:",
    pretrain_overlap_count,
)

group_overlap_rows = []
for fold, (fit_index, held_out_index) in enumerate(
    group_splitter.split(embeddings, groups=video_ids)
):
    held_videos = set(video_ids[held_out_index])
    group_overlap_rows.append({
        "fold": fold,
        "test_videos_in_train": int(
            len(held_videos & set(video_ids[fit_index]))
        ),
        "test_sequences_from_pretrain_video": int(
            np.isin(video_ids[held_out_index], normal_videos).sum()
        ),
    })
group_overlap_table = pd.DataFrame(group_overlap_rows)
display(group_overlap_table)

## Audit 4: one-versus-normal with leave-one-video-out

The five-class grouped split is structurally limited because every class but myopathic has one to three videos. A cleaner partial check is binary: condition versus normal, holding out one condition video at a time. The normal side still comes from one video, so this measures whether condition videos stay distinguishable from a fixed normal reference. Small video counts make these descriptive numbers, not clinical estimates.

In [ ]:
one_vs_normal_rows = []
for condition in ["parkinsons", "stroke", "cerebralpalsy", "myopathic"]:
    keep = np.isin(labels, [condition, "normal"])
    sub_index = np.flatnonzero(keep)
    sub_labels = (labels[keep] == condition).astype(int)
    sub_groups = video_ids[keep]
    condition_videos = sorted(set(sub_groups[sub_labels == 1]))
    fold_rows = []
    for held_video in condition_videos:
        fit = np.flatnonzero(sub_groups != held_video)
        held = np.flatnonzero(sub_groups == held_video)
        model = make_rf()
        model.fit(embeddings[sub_index[fit]], sub_labels[fit])
        prediction = model.predict(embeddings[sub_index[held]])
        fold_rows.append({
            "held_out_video": held_video,
            "test_sequences": int(len(held)),
            "accuracy": float(accuracy_score(sub_labels[held], prediction)),
        })
    table = pd.DataFrame(fold_rows)
    one_vs_normal_rows.append({
        "condition": condition,
        "videos": len(condition_videos),
        "mean_accuracy": float(table["accuracy"].mean()),
        "accuracy_std": float(table["accuracy"].std(ddof=1)),
        "min_accuracy": float(table["accuracy"].min()),
        "max_accuracy": float(table["accuracy"].max()),
    })
one_vs_normal_table = pd.DataFrame(one_vs_normal_rows)
display(one_vs_normal_table)

## A reporting template for movement encoders

The workshop asks how progress is measured and whether pretraining genuinely helps. This notebook supports a five-line reporting template for any pose or movement encoder paper:

1. Report the video-ID probe for the pretrained representation and a raw-coordinate baseline, with the chance line.
2. Report both a sequence split and a source-grouped split, with standard deviation across folds.
3. Report the majority-class and missingness controls computed in the same code.
4. Count test clips that share the pretraining source video.
5. Report the generalization gap as the headline honesty metric.

A movement representation is not bad merely because it encodes person identity; gait is personally identifying. It is questionable when it encodes more video identity than the raw pose already contains.

In [ ]:
import json

import matplotlib.pyplot as plt

probe_table.to_csv(ARTIFACT_DIR / "07_video_identity_probe.csv", index=False)
sequence_lane_table = pd.DataFrame([{
    "lane": "sequence_70_30_video_confounded",
    "test_sequences": len(test_index),
    **sequence_metrics,
}])
group_fold_table.to_csv(
    ARTIFACT_DIR / "07_video_grouped_five_class_folds.csv", index=False
)
sequence_fold_table.to_csv(
    ARTIFACT_DIR / "07_sequence_grouped_five_class_folds.csv", index=False
)
sequence_lane_table.to_csv(
    ARTIFACT_DIR / "07_sequence_lane_metrics.csv", index=False
)
with open(ARTIFACT_DIR / "07_generalization_gap.json", "w") as gap_file:
    json.dump({
        "generalization_gap": float(generalization_gap),
        "sequence_lane_accuracy": sequence_metrics["accuracy"],
        "video_grouped_mean_accuracy": float(
            group_fold_table["accuracy"].mean()
        ),
        "video_grouped_mean_std": float(
            group_fold_table["accuracy"].std(ddof=1)
        ),
        "sequence_grouped_mean_accuracy": float(
            sequence_fold_table["accuracy"].mean()
        ),
        "sequence_grouped_mean_std": float(
            sequence_fold_table["accuracy"].std(ddof=1)
        ),
        "matched_grouping_gap": float(matched_gap),
        "sequence_lane_majority": sequence_metrics["majority_accuracy"],
        "video_grouped_mean_majority": float(
            group_fold_table["majority_accuracy"].mean()
        ),
        "sequence_grouped_mean_majority": float(
            sequence_fold_table["majority_accuracy"].mean()
        ),
    }, gap_file, indent=2)
group_overlap_table.to_csv(
    ARTIFACT_DIR / "07_pretrain_overlap.csv", index=False
)
one_vs_normal_table.to_csv(
    ARTIFACT_DIR / "07_one_vs_normal_grouped.csv", index=False
)

figure, axes = plt.subplots(1, 2, figsize=(11.5, 4))
axes[0].bar(
    range(len(probe_table)),
    probe_table["mean_video_id_accuracy"],
    yerr=probe_table["accuracy_std"],
    color=["#ef7d57", "#c98d5e", "#9ac7bf", "#b9d6c6"],
)
axes[0].axhline(
    1.0 / len(unique_videos),
    color="#17324d", linestyle="--", label="chance",
)
axes[0].set_xticks(range(len(probe_table)))
axes[0].set_xticklabels(
    probe_table["representation"], rotation=12, ha="right"
)
axes[0].set_ylabel("video-ID accuracy (3 repeated 70/30 splits)")
axes[0].set_title("Audit 1: video identity in each representation")
axes[0].legend()

lane_labels = [
    "sequence split",
    "video-grouped (6 folds)",
    "sequence-grouped (6 folds)",
    "majority (sequence)",
    "majority (grouped)",
]
lane_values = [
    sequence_metrics["accuracy"],
    group_fold_table["accuracy"].mean(),
    sequence_fold_table["accuracy"].mean(),
    sequence_metrics["majority_accuracy"],
    group_fold_table["majority_accuracy"].mean(),
]
lane_errors = [
    0.0,
    group_fold_table["accuracy"].std(ddof=1),
    sequence_fold_table["accuracy"].std(ddof=1),
    0.0,
    0.0,
]
axes[1].bar(
    lane_labels, lane_values, yerr=lane_errors,
    color=["#ef7d57", "#17324d", "#5c806e", "#c8ccd4", "#8a97a5"],
)
axes[1].set_ylabel("five-class accuracy")
axes[1].set_title("Audit 2: sequence, video-grouped, and matched lanes")
axes[1].tick_params(axis="x", rotation=15)
figure.tight_layout()
figure.savefig(
    ARTIFACT_DIR / "07_source_identity_summary.png",
    dpi=160, bbox_inches="tight",
)
plt.show()
print("audit artifacts written to", ARTIFACT_DIR)

## What this means for the workshop paper

The strongest five-page story is not another accuracy table. It is the measured gap between what a sequence split promises and what a source-grouped split delivers, plus the probe evidence that the frozen encoder's identity content is, or is not, explainable by the pose itself. Notebook 08 probes whether the latents linearly encode biomechanical quantities, notebook 09 ablates the mask geometry, and notebook 10 turns the infilling objective into a causal future predictor.

If the grouped five-class accuracy collapses toward the majority line, that is a publishable negative result for a field that currently over-reports sequence splits on internet video.